In [1]:
!pip install -q gradio scikit-image
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import denoise_tv_chambolle
import gradio as gr
import random
print("Installation terminée")

✅ Installation terminée


In [2]:
# Charger MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train[..., np.newaxis] / 255.0
x_test = x_test[..., np.newaxis] / 255.0

# Modèle (si tu l'as déjà entraîné, tu peux sauter le fit)
model = keras.Sequential([
    keras.layers.Conv2D(32, 3, activation='relu', input_shape=(28,28,1)),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=3, batch_size=128, verbose=1)  # 3 epochs pour aller plus vite
print("Modèle prêt")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 59s 116ms/step - accuracy: 0.9370 - loss: 0.2174
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 48s 103ms/step - accuracy: 0.9829 - loss: 0.0562
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 48s 102ms/step - accuracy: 0.9875 - loss: 0.0409
✅ Modèle prêt


In [3]:
def fgsm_attack(image, epsilon=0.1):
    image = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        pred = model(image)
        loss = tf.keras.losses.sparse_categorical_crossentropy(tf.argmax(pred, axis=1), pred)
    gradient = tape.gradient(loss, image)
    perturbation = epsilon * tf.sign(gradient)
    adv_image = tf.clip_by_value(image + perturbation, 0, 1)
    return adv_image[0].numpy()

def numerical_inverse(adv_image, weight=0.15):
    restored = denoise_tv_chambolle(adv_image[..., 0], weight=weight, eps=1e-5)
    return np.expand_dims(restored, axis=-1)

print(" Fonctions FGSM et TV prêtes")

✅ Fonctions FGSM et TV prêtes


In [4]:
# Variables globales
current_clean = None
adv_image_global = None

def load_random_image():
    idx = random.randint(0, len(x_test)-1)
    img = x_test[idx]                    # (28,28,1) déjà normalisé
    true_label = int(y_test[idx])
    display_img = (img[..., 0] * 255).astype(np.uint8)
    return display_img, img, true_label

def process_clean():
    global current_clean
    display_img, model_img, true_label = load_random_image()
    current_clean = model_img

    pred = model.predict(model_img[np.newaxis,...], verbose=0)
    label = int(np.argmax(pred[0]))
    conf = float(np.max(pred[0])*100)
    return f" Prédiction : {label} ({conf:.1f}%) | Vrai : {true_label}", display_img

def process_attack(epsilon=0.1):
    global current_clean, adv_image_global
    if current_clean is None:
        return " Clique d'abord sur Prédire", None
    adv = fgsm_attack(current_clean[...,0], epsilon=epsilon)
    adv_image_global = adv[..., np.newaxis]

    pred = model.predict(adv_image_global[np.newaxis,...], verbose=0)
    label = int(np.argmax(pred[0]))
    conf = float(np.max(pred[0])*100)
    display_adv = (adv * 255).astype(np.uint8)
    return f" Attaque → {label} ({conf:.1f}%)", display_adv

def process_numerical(weight=0.15):
    global adv_image_global
    if adv_image_global is None:
        return " Fais d'abord l'attaque !", None
    restored = numerical_inverse(adv_image_global, weight=weight)
    pred = model.predict(restored[np.newaxis,...], verbose=0)
    label = int(np.argmax(pred[0]))
    conf = float(np.max(pred[0])*100)
    display_rest = (restored[...,0] * 255).astype(np.uint8)
    return f" Restauration → {label} ({conf:.1f}%)", display_rest

# Interface
with gr.Blocks() as demo:
    gr.Markdown("# Détection Adversariale par Problème Inverse (MNIST)")

    with gr.Row():
        btn_load = gr.Button("1. Charger Nouvelle Image Aléatoire", variant="primary")

    with gr.Row():
        btn_clean = gr.Button("2. Prédire (Image Propre)", variant="primary")
        output_clean = gr.Label(label="Résultat Prédiction")
        clean_plot = gr.Image(label="Image Propre")

    with gr.Row():
        epsilon_slider = gr.Slider(0.05, 0.5, value=0.1, step=0.05, label="Epsilon (Force Attaque)")
        weight_slider = gr.Slider(0.01, 0.5, value=0.15, step=0.01, label="Poids TV (Denoising)")

    with gr.Row():
        btn_attack = gr.Button("3. Lancer Attaque (FGSM)", variant="stop")
        output_attack = gr.Label(label="Résultat Attaque")
        adv_plot = gr.Image(label="Image Adversariale")

    with gr.Row():
        btn_analyze = gr.Button("4. Analyse Numérique (TV)", variant="primary")
        output_restored = gr.Label(label="Résultat Restauration")
        restored_plot = gr.Image(label="Image Restaurée")

    # Connexions
    btn_load.click(lambda: load_random_image()[0], outputs=clean_plot)
    btn_clean.click(process_clean, outputs=[output_clean, clean_plot])
    btn_attack.click(process_attack, inputs=epsilon_slider, outputs=[output_attack, adv_plot])
    btn_analyze.click(process_numerical, inputs=weight_slider, outputs=[output_restored, restored_plot])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ff4e284d7e3549258.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
